# Optimization & Walk-Forward Validation

Parameter search, cross-validation, and factor IC analysis using `quant.engine`.

In [ ]:
import pandas as pd; import numpy as np
from engine import Strategy, Signal, Engine, BacktestConfig, DataFrameSource, summary
from engine.optimize import GridSearch, RandomSearch
from engine.walkforward import WalkForward
from engine.factors import Factor, compute_ic, factor_returns
from engine.report import report_generate
print("All imports OK")

## 1. Prepare Data

In [ ]:
np.random.seed(42)
n = 504
dates = pd.date_range("2024-01-01", periods=n, freq="D")
close = pd.DataFrame({
    "SPY": 450 + np.cumsum(np.random.randn(n)*1.5+0.03),
    "QQQ": 380 + np.cumsum(np.random.randn(n)*2.0+0.04),
    "IWM": 200 + np.cumsum(np.random.randn(n)*1.8+0.01),
    "XLE": 85 + np.cumsum(np.random.randn(n)*1.2+0.02),
}, index=dates)
data = DataFrameSource(close=close)
cfg = BacktestConfig(initial_capital=100_000, slippage_bps=5, commission_bps=1)
print(f"Data: {data.close.shape}")

## 2. Define Strategy

In [ ]:
class MACrossover(Strategy):
    fast: int = 20; slow: int = 50
    def on_init(self, ctx):
        self.ma_fast = ctx.data.close.rolling(self.fast).mean()
        self.ma_slow = ctx.data.close.rolling(self.slow).mean()
    def on_bar(self, ctx, bar):
        if bar < self.slow: return []
        signals = []
        for sym in ctx.universe:
            if self.ma_fast.iloc[bar][sym] > self.ma_slow.iloc[bar][sym]:
                if not ctx.portfolio.has_position(sym):
                    signals.append(Signal.buy(sym, weight=1.0/len(ctx.universe)))
            else:
                if ctx.portfolio.has_position(sym):
                    signals.append(Signal.close(sym))
        return signals

## 3. GridSearch — Find Best Parameters

In [ ]:
gs = GridSearch(MACrossover,
    param_grid={"fast": [10, 20, 30, 50], "slow": [40, 60, 100, 150]},
    data=data, config=cfg, metric="sharpe_ratio")
results = gs.run()

print("Top 5 parameter combinations:")
for i, (params, metrics) in enumerate(results[:5]):
    print(f"  {i+1}. fast={params['fast']}, slow={params['slow']} "
          f"Sharpe={metrics['sharpe_ratio']:.2f} "
          f"Return={metrics['annual_return']:.2%} MaxDD={metrics['max_drawdown']:.2%}")

## 4. Walk-Forward — Validate Out-of-Sample

In [ ]:
best_params, _ = results[0]
strategy = MACrossover()
strategy.fast = best_params['fast']
strategy.slow = best_params['slow']

wf = WalkForward(strategy, data, cfg, train_window="6M", test_window="1M")
oos = wf.summary()
print("Walk-forward out-of-sample performance:")
for k, v in oos.items():
    if k != "n_folds":
        print(f"  {k}: {v:.4f}")
print(f"  Folds: {oos['n_folds']}")

# Check consistency across folds
folds = wf.run()
sharpe_by_fold = [f["test_metrics"]["sharpe_ratio"] for f in folds]
print(f"\nOOS Sharpe by fold: {[f'{s:.2f}' for s in sharpe_by_fold]}")
print(f"Mean: {np.mean(sharpe_by_fold):.2f}, Std: {np.std(sharpe_by_fold):.2f}")
if np.mean(sharpe_by_fold) > 0 and np.mean(sharpe_by_fold)/max(np.std(sharpe_by_fold), 0.001) > 1:
    print("Strategy is robust (positive average Sharpe, reasonable consistency)")
else:
    print("Warning: strategy may not be robust out-of-sample")

## 5. Factor IC Analysis

In [ ]:
momentum = Factor("momentum", lambda df: df.pct_change(60))
volatility = Factor("volatility", lambda df: df.pct_change().rolling(20).std())

fwd_5d = data.close.pct_change(5).shift(-5)
ic_results = compute_ic([momentum, volatility], fwd_5d, data)

for name, ic_series in ic_results.items():
    if len(ic_series) > 0:
        mean_ic = ic_series.mean()
        ic_ir = mean_ic / ic_series.std() if ic_series.std() > 0 else 0
        print(f"{name}: mean IC={mean_ic:.4f}, IC IR={ic_ir:.2f}, "
              f"hit rate={(ic_series>0).mean():.1%}")

# Plot IC over time
import matplotlib.pyplot as plt
fig, axes = plt.subplots(len(ic_results), 1, figsize=(12, 3*len(ic_results)))
if len(ic_results) == 1: axes = [axes]
for ax, (name, ic) in zip(axes, ic_results.items()):
    if len(ic) > 0:
        ax.plot(ic.index, ic.values, linewidth=0.5, color="#1a1a2e")
        ax.axhline(y=0, color="red", linestyle="--", linewidth=0.5)
        ax.set_title(f"{name} IC")
        ax.set_ylabel("Spearman Rank IC")
plt.tight_layout()
plt.show()

## 6. RandomSearch — Larger Parameter Space

In [ ]:
rs = RandomSearch(MACrossover,
    param_grid={"fast": list(range(5, 100, 5)), "slow": list(range(20, 200, 10))},
    data=data, config=cfg, n_iter=50, metric="sharpe_ratio")
results_rs = rs.run()

best_p, best_m = results_rs[0]
print(f"Best random: fast={best_p['fast']}, slow={best_p['slow']}")
print(f"Sharpe={best_m['sharpe_ratio']:.2f}, Return={best_m['annual_return']:.2%}")

# Plot search space
fast_vals = [r[0]["fast"] for r in results_rs]
slow_vals = [r[0]["slow"] for r in results_rs]
sharpes = [r[1]["sharpe_ratio"] for r in results_rs]
plt.figure(figsize=(8, 6))
sc = plt.scatter(fast_vals, slow_vals, c=sharpes, cmap="RdYlGn", s=60)
plt.colorbar(sc, label="Sharpe")
plt.xlabel("fast"); plt.ylabel("slow")
plt.title("RandomSearch: Sharpe by Parameter")
plt.show()

## 7. Generate Final Report

In [ ]:
best_strategy = MACrossover()
best_strategy.fast = best_p['fast']
best_strategy.slow = best_p['slow']
final_result = Engine(cfg).run(best_strategy, data)
report_generate(final_result, "optimized_ma_crossover_report.html")
print("Final report saved to optimized_ma_crossover_report.html")

## Key Takeaways

1. **GridSearch** finds the best parameters in-sample — but don't trust these numbers alone.
2. **Walk-forward** validates whether the strategy works out-of-sample. If OOS Sharpe is consistently positive across folds, you have a robust strategy.
3. **Factor IC** measures whether your signal has predictive power. |IC| > 0.05 with IC IR > 0.5 is a good target.
4. **Always report OOS performance** — in-sample metrics are optimistic.